In [ ]:
# CELL 1 — Install YOLO and dependencies
!pip install -q ultralytics opencv-python tqdm

import os
import cv2
import torch
from ultralytics import YOLO
from google.colab import files
from tqdm import tqdm

print("Torch CUDA available:", torch.cuda.is_available())

Torch CUDA available: True


In [ ]:
# CELL 2 — Upload a video from your desktop
uploaded = files.upload()

# Assume single file upload
video_filename = list(uploaded.keys())[0]
print("Uploaded video:", video_filename)

Saving input.mp4.mp4 to input.mp4 (1).mp4
Uploaded video: input.mp4 (1).mp4


In [ ]:
# CELL 3 — Load YOLOv9-Seg model
# You can switch to another seg model if desired, e.g. 'yolov8x-seg.pt'
model = YOLO("yolov9c-seg.pt")  # if this fails, try: "yolov8x-seg.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Model loaded on:", device)

Model loaded on: cuda


In [ ]:
# CELL 4 — Run instance segmentation on the video (patched for mask resizing)

input_path = video_filename
output_path = "segmented_output.mp4"

# Open input video
cap = cv2.VideoCapture(input_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {width}x{height} @ {fps} FPS, {frame_count} frames")

# Define video writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# COCO class IDs for people, bikes, vehicles:
# person=0, bicycle=1, car=2, motorcycle=3, bus=5, truck=7
target_ids = {0, 1, 2, 3, 5, 7}

for _ in tqdm(range(frame_count), desc="Processing frames"):
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO segmentation
    results = model.predict(
        frame,
        device=device,
        verbose=False,
        conf=0.4
    )

    annotated = frame.copy()

    for r in results:
        if r.masks is None:
            continue

        boxes = r.boxes
        masks = r.masks.data.cpu().numpy()  # [N, H_mask, W_mask]

        for i, box in enumerate(boxes):
            cls_id = int(box.cls.item())
            if cls_id not in target_ids:
                continue

            # --- FIX: Resize mask to match the frame resolution ---
            mask = masks[i]
            mask_resized = cv2.resize(mask.astype("float32"), (width, height))

            # Apply mask overlay
            colored = annotated.copy()
            colored[mask_resized > 0.5] = (0, 255, 0)  # green overlay
            annotated = cv2.addWeighted(colored, 0.5, annotated, 0.5, 0)

            # Bounding box + label
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            conf = float(box.conf.item())
            label = f"{model.names[cls_id]} {conf:.2f}"

            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                annotated,
                label,
                (x1, max(0, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                1,
                cv2.LINE_AA
            )

    out.write(annotated)

cap.release()
out.release()
print("Saved segmented video to:", output_path)

Video: 1920x1080 @ 23.793868943571464 FPS, 1731 frames


Processing frames: 100%|██████████| 1731/1731 [02:44<00:00, 10.51it/s]

Saved segmented video to: segmented_output.mp4


In [ ]:
# CELL 5 — Download the output video
from google.colab import files
files.download("segmented_output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>